# OpenXAI-based synthetic data generator

In [ ]:
import sys

import numpy as np
import pandas as pd

!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [11]:
import openxai

# Utils
import torch

# Data loaders
from openxai.dataloader import return_loaders

In [19]:
# RETURN two DataFrames with data and labels generated with OpenXAI

def synthetic_generator(n_samples, dimensions):
    
    data_name='synthetic'

    gauss_params= {'n_samples': n_samples,
                   'dim': dimensions,
                   'n_clusters': 10,
                   'distance_to_center': 5,
                   'test_size': 0.20,
                   'upper_weight': 1,
                   'lower_weight': -1,
                   'seed': 564,
                   'sigma': None,
                   'sparsity': 0.25
                   }

    # Get training and test loaders
    loader_train, loader_test= return_loaders(data_name=data_name, 
                                              download=True, 
                                              batch_size=n_samples, 
                                              scaler='minmax', 
                                              gauss_params=gauss_params)

    data_it= iter(loader_train)

    inputs, labels, sy_weights, sy_masks, sy_masked_weights, sy_probs, sy_cluster_idx= data_it.next()
    labels= labels.type(torch.int64)


    # convert OpenXAI inputs and labels from tensor to pd.DataFrame as required by T-Exp methods
    n_fts= inputs.shape[1]
    ft_names= []

    for i in range(n_fts):
        ft_names.append("ft_" + str(i+1))

    df_inputs= pd.DataFrame(data=inputs, columns=ft_names)   # OpenXAI synthetic data is already normalized
    df_labels= pd.DataFrame(data=labels, columns=["target"])


    return df_inputs, df_labels

In [20]:
x_data, y_labels= synthetic_generator(100, 20)

x_data.shape

(100, 20)

In [21]:
x_data.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
0,0.145010,0.339611,0.196211,0.203350,0.166244,0.096349,0.775523,0.387130,0.249836,0.277686,0.518756,0.232638,0.616792,0.553426,0.373845,0.622479,0.597430,0.412096,0.697586,0.501481
1,0.284304,0.280112,0.869580,0.275309,0.319706,0.343733,0.201777,0.381086,0.163085,0.271164,0.251296,0.315474,0.469995,0.487431,0.466634,0.517754,0.542271,0.652768,0.575941,0.382945
2,0.201864,0.358450,0.389542,0.320625,0.122708,0.088974,0.313841,0.116088,0.320288,0.787016,0.637712,0.458781,0.622398,0.580371,0.439267,0.306483,0.446750,0.884054,0.616672,0.319126
3,0.282630,0.207587,0.347179,0.668542,0.224740,0.187641,0.283535,0.335869,0.373722,0.462274,0.656081,0.625652,0.522983,0.720842,0.791238,0.544626,0.359050,0.362973,0.581640,0.321974
4,0.371273,0.723751,0.343272,0.263171,0.118510,0.237584,0.137105,0.303181,0.471611,0.437520,0.407754,0.615933,0.483341,0.418950,0.508797,0.635104,0.521458,0.705326,0.722884,0.243031


In [22]:
y_labels.head()

,target
0,1
1,1
2,0
3,1
4,1


In [2]:
# save a pd.DataFrame dataset in a csv file

def save_dataset(df_x, df_y, file_name='synth_data.csv'):
    
    df= pd.concat([df_x, df_y], axis=1)

    lines= df.shape[0]
    colls= df.shape[1]

    f= open(file_name, 'w')

    line= str(df.columns[0])

    for i in range((colls-1)):
        line= line+','+str(df.columns[(i+1)])

    line= line+'\n'
    f.write(line)

    df_array= np.asarray(df)
    line= str(df.columns[0])

    for i in range(lines):
        elements= df_array[i]
        line= str(elements[0])

        for j in range((colls-1)):
            line= line+','+str(elements[(j+1)])

        line= line+'\n'
        f.write(line)

    f.close()